# Lekcja 08 - Wzorzec projektowy Multi-Agent


## Konfiguracja


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dlaczego systemy wieloagentowe?

Zadania w rzeczywistym świecie, takie jak planowanie podróży, obejmują wiele różnych dziedzin wiedzy — logistykę, znajomość lokalną, budżetowanie i inne. Pojedynczy agent próbujący poradzić sobie ze wszystkim szybko staje się nieporęczny.

Systemy wieloagentowe rozwiązują ten problem poprzez **specjalizację**: każdy agent koncentruje się na jednej dziedzinie, co daje lepsze rezultaty niż ogólne podejście. Poprawiają także **skalowalność** — można dodać nowych agentów (np. specjalistę od lotów, krytyka kulinarnego) bez konieczności przerabiania istniejącego przepływu pracy. Agenci współpracują poprzez uporządkowany proces, przekazując kontekst z jednego do drugiego.


## Tworzenie specjalistycznych agentów


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Budowanie sekwencyjnego przepływu pracy

`WorkflowBuilder` pozwala na połączenie agentów w skierowany graf. Tutaj tworzymy prostą dwustopniową linię działania: **TravelPlanner** opracowuje plan podróży, a następnie **TravelConcierge** go przegląda i ulepsza.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodawanie kolejnych agentów do przepływu pracy

Jedną z największych zalet wzorca wieloagentowego jest łatwość rozbudowy. Poniżej dodajemy agenta **BudgetReviewer**, który sprawdza plan względem budżetu podróżnego, oznacza pozycje mogące przekroczyć limit kosztów i sugeruje oszczędne alternatywy. Przepływ pracy uruchamia teraz trzech agentów kolejno: 

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Podsumowanie

W tej lekcji nauczyłeś się, jak:

1. **Tworzyć wyspecjalizowanych agentów** — każdy z określoną rolą (planowanie, concierge, przegląd budżetu).
2. **Łączyć agentów w sekwencyjny przepływ pracy** używając `WorkflowBuilder` i `add_edge`.
3. **Streamować output** z wieloagentowego pipeline’u, śledząc który agent mówi.
4. **Rozszerzać przepływ pracy** dodając nowych agentów do łańcucha bez modyfikowania istniejących.

Wzorzec projektowy wieloagentowy utrzymuje każdego agenta prostym, jednocześnie produkując bogatsze, bardziej dokładnie zweryfikowane wyniki niż mógłby osiągnąć pojedynczy agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mimo że dokładamy starań, aby tłumaczenie było jak najbardziej precyzyjne, prosimy mieć na uwadze, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego oryginalnym języku powinien być uważany za źródło wiarygodne. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
